
# Projet Home Credit — Sujet 03

**Séance :** 01  
**Nom :** YOUSOUFOU Malikath Reussite  
**Porteur :** P1  

**Étapes assignées :**  
1.1 : inventorier les tables  
1.7 : doublons  
1.8 : fuites  
1.11 : la métrique

In [15]:
import glob, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

d = glob.glob("/kaggle/input/**/train.csv", recursive=True)[0].replace("train.csv", "")
train = pd.read_csv(d + "train.csv")
a_decider = pd.read_csv(d + "a_decider.csv")
dico = pd.read_csv(d + "HomeCredit_columns_description.csv", encoding="latin-1")

In [16]:
## Étape 1.1 — Inventorier les tables


for nom, table in [("train.csv", train), ("a_decider.csv", a_decider),
                   ("HomeCredit_columns_description.csv", dico)]:
    print(nom, "->", table.shape[0], "lignes,", table.shape[1], "colonnes")

train.csv -> 246008 lignes, 122 colonnes
a_decider.csv -> 61503 lignes, 121 colonnes
HomeCredit_columns_description.csv -> 219 lignes, 5 colonnes


In [17]:
## Étape 1.8 — Fuites
a_ecarter = {
    "TARGET": "c'est la cible : l'issue du prêt n'existe pas le jour de la décision",
    "SK_ID_CURR": "identifiant interne, sans signification métier",
    "FLAG_MOBIL": "constante (toujours 1) : aucune information",
}
for colonne, raison in a_ecarter.items():
    print(colonne, "->", raison)

X = train.drop(columns=list(a_ecarter))
print("Colonnes conservées :", X.shape[1])

TARGET -> c'est la cible : l'issue du prêt n'existe pas le jour de la décision
SK_ID_CURR -> identifiant interne, sans signification métier
FLAG_MOBIL -> constante (toujours 1) : aucune information
Colonnes conservées : 119


In [18]:
## Étape 1.7 — Doublons
nb_doublons = train.drop(columns="SK_ID_CURR").duplicated().sum()
print("Lignes en double :", nb_doublons)
if nb_doublons == 0:
    print("Décision : on ne retire rien (aucun doublon).")
else:
    train = train.drop_duplicates(subset=train.columns.drop("SK_ID_CURR"))
    print("Décision : doublons retirés, il reste", len(train), "lignes.")

Lignes en double : 0
Décision : on ne retire rien (aucun doublon).


In [19]:
## Étape 1.11 — Métrique
C_FN, C_FP = 10, 1   
y = train.TARGET.values

def cout_moyen(refus):
    """refus = 1 si on refuse la demande, 0 si on l'accorde."""
    defauts_accordes = ((refus == 0) & (y == 1)).sum()
    bons_refuses = ((refus == 1) & (y == 0)).sum()
    return (C_FN * defauts_accordes + C_FP * bons_refuses) / len(y)

print("Accorder tout :", cout_moyen(np.zeros(len(y))), "| exactitude", (y == 0).mean())
print("Refuser tout  :", cout_moyen(np.ones(len(y))), "| exactitude", (y == 1).mean())

score = 1 - train.EXT_SOURCE_2.fillna(train.EXT_SOURCE_2.median())
print("AUC :", roc_auc_score(y, score), "| PR-AUC :", average_precision_score(y, score))
print("Meilleur coût et part refusée :",
      min((cout_moyen((score >= score.quantile(1 - p)).values), p) for p in np.linspace(0.01, 0.6, 60)))

Accorder tout : 0.8072908198107379 | exactitude 0.9192709180189262
Refuser tout  : 0.9192709180189262 | exactitude 0.08072908198107379
AUC : 0.6543994409564262 | PR-AUC : 0.15106765764985347
Meilleur coût et part refusée : (np.float64(0.6644418067705116), np.float64(0.33))
